# Fine-tuning: Dev-tuning 

Цель ноутбука — быстро и обоснованно выбрать:
•	1 основной fine-tuning сценарий
•	1 фиксированный набор гиперпараметров
для последующего полного эксперимента (Dev/Test).

**Общая стратегия**
Двухэтапный Dev-tuning:
Фаза 1 — сравнение сценариев (coarse selection)
→ выбираем 2 лучших сценария
Фаза 2 — мини-тюнинг гиперпараметров (refinement)
→ выбираем финальный протокол


## 1. Импорты

In [1]:
import os
import json
import math
import random
import gc
from pathlib import Path
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

import stage5_utils as u

import model_unet as m

## 2. Конфиги

In [2]:
RUNTIME_CONFIG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pin_memory": torch.cuda.is_available(),

    "batch_size": 64,
    "num_workers": 2,

    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "fallback_p_for_zero": 10,
    "val_ratio": 0.2,
}

In [3]:
PATHS = {
    "data_root": Path("/kaggle/input/datasets/taisiyaglazova"),
    "encoder_checkpoint": Path("/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt"),
    "results_root": Path("/kaggle/working/stage5_results"),
}

In [4]:
DEV_TUNING_CONFIG = {
    "tag": "stage5_block4_dev_tuning",
    "subjects": ["subj_054", "subj_065", "subj_090", "subj_094"],
    "group": "benchmark",
    "p_list": [10, 40, 100],

    "scenarios": ["full_ft", "low_lr_encoder", "partial_ft", "warmup"],

    "primary_metric": "auc",
    "control_metric": "f1",
}

In [5]:
PHASE1_CONFIG = {
    "lr_encoder": 1e-5,
    "lr_head": 1e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 3,
}

PHASE2_GRID = {
    "lr_encoder": [1e-5, 3e-5],
    "lr_head": [1e-4, 3e-4],
    "weight_decay": [1e-4, 1e-3],
    "warmup_epochs": [3],
}

In [7]:
RUN_DIR = PATHS["results_root"] / DEV_TUNING_CONFIG["tag"]

ARTIFACT_DIRS = {
    "root": RUN_DIR,
    "phase1": RUN_DIR / "phase1",
    "phase2": RUN_DIR / "phase2",
    "history": RUN_DIR / "history",
    "predictions": RUN_DIR / "predictions",
    "tables": RUN_DIR / "tables",
    "figures": RUN_DIR / "figures",
}


for path in ARTIFACT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

In [8]:
def make_param_grid(grid_dict):
    keys = list(grid_dict.keys())
    values = [grid_dict[k] for k in keys]
    return [dict(zip(keys, combo)) for combo in product(*values)]

PHASE2_PARAM_GRID = make_param_grid(PHASE2_GRID)

In [9]:
# Sanity print
print("Dev-tuning configuration loaded.\n")

print("=== Experiment setup ===")
print(f"Tag           : {DEV_TUNING_CONFIG['tag']}")
print(f"Subjects      : {DEV_TUNING_CONFIG['subjects']}")
print(f"Group         : {DEV_TUNING_CONFIG['group']}")
print(f"p list        : {DEV_TUNING_CONFIG['p_list']}")
print(f"Scenarios     : {DEV_TUNING_CONFIG['scenarios']}")
print(f"Primary metric: {DEV_TUNING_CONFIG['primary_metric']}")
print(f"Control metric: {DEV_TUNING_CONFIG['control_metric']}\n")

print("=== Runtime ===")
print(f"Device        : {RUNTIME_CONFIG['device']}")
print(f"Batch size    : {RUNTIME_CONFIG['batch_size']}")
print(f"Epochs        : {RUNTIME_CONFIG['max_epochs']}")
print(f"Patience      : {RUNTIME_CONFIG['patience']}\n")

print("=== Phase configs ===")
print(f"Phase 1 config: {PHASE1_CONFIG}")
print(f"Phase 2 grid size: {len(PHASE2_PARAM_GRID)}\n")

print("=== Paths ===")
print(f"Results dir   : {RUN_DIR}")
print(f"Data root     : {PATHS['data_root']}")
print(f"Encoder ckpt  : {PATHS['encoder_checkpoint']}")

Dev-tuning configuration loaded.

=== Experiment setup ===
Tag           : stage5_block4_dev_tuning
Subjects      : ['subj_054', 'subj_065', 'subj_090', 'subj_094']
Group         : benchmark
p list        : [10, 40, 100]
Scenarios     : ['full_ft', 'low_lr_encoder', 'partial_ft', 'warmup']
Primary metric: auc
Control metric: f1

=== Runtime ===
Device        : cpu
Batch size    : 64
Epochs        : 100
Patience      : 10

=== Phase configs ===
Phase 1 config: {'lr_encoder': 1e-05, 'lr_head': 0.0001, 'weight_decay': 0.0001, 'warmup_epochs': 3}
Phase 2 grid size: 8

=== Paths ===
Results dir   : /kaggle/working/stage5_results/stage5_block4_dev_tuning
Data root     : /kaggle/input/datasets/taisiyaglazova
Encoder ckpt  : /kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt


## 3. Воспроизводимость

In [10]:
u.set_seed(RUNTIME_CONFIG["seed"])
print("Device:", RUNTIME_CONFIG["device"])

Device: cpu


## 4. Пути

In [11]:
# Пути к датасетам
DATASETS = {
    "bigp3_train": PATHS["data_root"] / "bigp3bci-downstream-train",
    "bigp3_benchmark": PATHS["data_root"] / "bigp3bci-downstream-benchmark",
    "bcicomp3": PATHS["data_root"] / "bcicompiii-ds2",  
}

In [12]:
GROUPS = ["train", "benchmark", "bcicomp3"]

### Автореестр субъектов

In [13]:
def list_subject_ids(group: str):
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        data_dir = DATASETS["bigp3_train"] / "train"
    elif group == "benchmark":
        data_dir = DATASETS["bigp3_benchmark"]/ "benchmark"

    subject_ids = sorted([p.stem for p in data_dir.glob("*.npz")])
    return subject_ids

In [14]:
# Сборка реестра
SUBJECT_REGISTRY = {
    "train": list_subject_ids("train"),
    "benchmark": list_subject_ids("benchmark"),
}

print("Train subjects:", len(SUBJECT_REGISTRY["train"]))
print("Benchmark subjects:", len(SUBJECT_REGISTRY["benchmark"]))
print("Example train:", SUBJECT_REGISTRY["train"][:5])
print("Example benchmark:", SUBJECT_REGISTRY["benchmark"][:5])


Train subjects: 93
Benchmark subjects: 65
Example train: ['subj_000', 'subj_001', 'subj_002', 'subj_003', 'subj_004']
Example benchmark: ['subj_051', 'subj_052', 'subj_053', 'subj_054', 'subj_055']


In [15]:
# Валидация существования файлов
def check_subject_files(subject_id: str, group: str, p_list=(10, 20, 40, 60, 100)):
    report = {
        "epochs": u.get_epochs_path(subject_id, group).exists(),
        "split": u.get_split_path(subject_id, group).exists(),
        "stats": {p: u.get_stats_path(subject_id, p, group).exists() for p in p_list},
    }
    return report

example_subj = SUBJECT_REGISTRY["train"][80]
check_subject_files(example_subj, "train")

{'epochs': True,
 'split': True,
 'stats': {10: True, 20: True, 40: True, 60: True, 100: True}}

## Запуск Phase 1

Ячейки закомментированы, потому что результаты уже получены и повторный запуск не нужен

In [ ]:
# phase1_benchmark_root = ARTIFACT_DIRS["phase1"] / "benchmark_dev"

# phase1_benchmark_df = u.run_many(
#     subject_list=DEV_TUNING_CONFIG['subjects'],
#     p_list=DEV_TUNING_CONFIG['p_list'],
#     scenario_list=["ssl_ft"],
#     group=DEV_TUNING_CONFIG["group"],
#     runtime_config=RUNTIME_CONFIG,
#     results_root=phase1_benchmark_root,
#     encoder_checkpoint=PATHS["encoder_checkpoint"],
#     ft_strategy_list=DEV_TUNING_CONFIG["scenarios"],
#     lr_encoder=PHASE1_CONFIG["lr_encoder"],
#     lr_head=PHASE1_CONFIG["lr_head"],
#     weight_decay=PHASE1_CONFIG["weight_decay"],
#     warmup_epochs=PHASE1_CONFIG["warmup_epochs"],
#     save_history=True,
#     save_predictions=True,
#     save_summary_every=1,
#     continue_on_error=True,
# )

In [ ]:
# phase1_benchmark_df[[
#     "subject_id", "p", "ft_strategy", "selected_threshold",
#     "auc", "f1", "precision", "recall"
# ]]

In [ ]:
# # Для anchor-запуска (Subj A)
# phase1_anchor_root = ARTIFACT_DIRS["phase1"] / "anchor_bcicomp3"

# phase1_anchor_df = u.run_many(
#     subject_list=["subjA"],
#     p_list=DEV_TUNING_CONFIG["p_list"],
#     scenario_list=["ssl_ft"],
#     group="bcicomp3",
#     runtime_config=RUNTIME_CONFIG,
#     results_root=phase1_anchor_root,
#     encoder_checkpoint=PATHS["encoder_checkpoint"],
#     ft_strategy_list=DEV_TUNING_CONFIG["scenarios"],
#     lr_encoder=PHASE1_CONFIG["lr_encoder"],
#     lr_head=PHASE1_CONFIG["lr_head"],
#     weight_decay=PHASE1_CONFIG["weight_decay"],
#     warmup_epochs=PHASE1_CONFIG["warmup_epochs"],
#     save_history=True,
#     save_predictions=True,
#     save_summary_every=1,
#     continue_on_error=True,
# )

In [21]:
TOP_FT_STRATEGIES = ["partial_ft", "warmup"]

## Запуск Phase 2

In [ ]:
phase2_benchmark_root = ARTIFACT_DIRS["phase2"] / "benchmark_dev"

phase2_benchmark_df = u.run_many(
    subject_list=DEV_TUNING_CONFIG["subjects"],
    p_list=DEV_TUNING_CONFIG["p_list"],
    scenario_list=["ssl_ft"],
    group=DEV_TUNING_CONFIG["group"],
    runtime_config=RUNTIME_CONFIG,
    results_root=phase2_benchmark_root,
    encoder_checkpoint=PATHS["encoder_checkpoint"],
    ft_strategy_list=TOP_FT_STRATEGIES,
    param_grid=PHASE2_PARAM_GRID,
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

In [ ]:
phase2_summary = (
    phase2_benchmark_df
    .groupby(["ft_strategy", "lr_encoder", "lr_head", "weight_decay", "p"], as_index=False)
    .agg(
        mean_auc=("auc", "mean"),
        std_auc=("auc", "std"),
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),
    )
    .sort_values(["p", "mean_auc"], ascending=[True, False])
)

phase2_summary